In [2]:
%matplotlib widget

from blackjack_py import ProbabilisticRankShoe
from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
from datetime import datetime
import os
import tqdm
from collections import deque
# from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.floor_ceil_node import FloorCeilNode, SplitNode, DecisionNode, DealerCheckBJNode
from blackjack.dealer_sim import (
    run_dealer_cards_simulation,
    run_dealer_cards_simulation_recursive,
    load_realistic_combs_with_counts,
    _dealer_stand_or_bust,
    _run_dealer_cards_simulation
)
from blackjack.hand import ValueOnlyHand
from collections import defaultdict
import datetime
import matplotlib.pyplot as plt
from blackjack.tree_utils import iterate_nodes_by_levels
import os
from blackjack import abstract_node

In [3]:
def run_dealer_cards_simulation_by_outcome(
    bj_round: BJRound,
    shoe: ProbabilisticRankShoe,
    n_dealer_sim_runs: int,
    depth_cutoff,
    reset_shoe_sampler: bool = True
):
    """Run Monte Carlo simulations for dealer play."""
    values = []
    dealer_bust_outcome = []
    dealer_stand_outcome = []
    dealer_other_outcome = []

    if shoe.is_dealer_card_locked():
        shoe.unlock_dealer_card()
        shoe = shoe.copy()
    
    for i in range(n_dealer_sim_runs):
        bj_round_copy = bj_round.copy()
        shoe_copy = shoe.copy()
        if reset_shoe_sampler:
            shoe_copy.change_random_sampler()

        # Simulate dealer cards until round over
        while not bj_round_copy.get_stage() == BJStage.ROUND_OVER:
            possible_values = bj_round_copy.get_possible_next_card_ranks()
            card = shoe_copy.sample_and_burn_rank(possible_values)
            bj_round_copy.take_card(card)
        
        if len(bj_round.dealer_hand) >= depth_cutoff:
            dealer_other_outcome.append(bj_round_copy.get_player_value())
        elif bj_round_copy.dealer_hand.is_bust():
            dealer_bust_outcome.append(bj_round_copy.get_player_value())
        else:
            dealer_stand_outcome.append(bj_round_copy.get_player_value())

        # Collect results
        values.append(bj_round_copy.get_player_value())
    return values, dealer_bust_outcome, dealer_stand_outcome, dealer_other_outcome

In [411]:
"abc"[:-1]

'ab'

In [422]:
from collections import Counter
from math import factorial
from itertools import product
from copy import copy


def get_final_combo(cards, upcard, dealer_no_blackjack):
    if dealer_no_blackjack and upcard == 11 and not set(cards) == {10}:
        cards = tuple(sorted(cards))
        # move tens to the back if possible
        while cards[0] == 10:
            cards = cards[1:] + (10,)
        return cards
    else:
        return tuple(sorted(cards))


def _get_realistic_combinations_from_previous(
    depth, upcard, dealer_hit_soft_17, dealer_no_blackjack,
    previous_depth_data   
):
    bust_counts = copy(previous_depth_data["bust_counts"])
    stand_counts = copy(previous_depth_data["stand_counts"])
    other_counts = defaultdict(int)  # copy(previous_depth_data["other_counts"])

    for combo in product(range(2, 12), repeat=depth):
        if upcard + combo[0] == 21 and dealer_no_blackjack:
            continue
        hand = ValueOnlyHand([upcard])
        stopped_early = False

        for v in combo:
            if _dealer_stand_or_bust(hand, dealer_hit_soft_17):
                stopped_early = True
                break
            hand.add_card(v)
        if stopped_early:
            continue

        final_combo = get_final_combo(hand.cards[1:], upcard, dealer_no_blackjack)
        if hand.is_bust():
            bust_counts[final_combo] += 1
        elif _dealer_stand_or_bust(hand, dealer_hit_soft_17):
            stand_counts[final_combo] += 1
        else:
            other_counts[final_combo] += 1
            
                
    return _format_output(bust_counts, stand_counts, other_counts, upcard)


def get_realistic_combinations(
        depth, upcard, dealer_hit_soft_17, dealer_no_blackjack,
        previous_depth_data=None
    ):
    if previous_depth_data is not None:
        return _get_realistic_combinations_from_previous(
            depth, upcard, dealer_hit_soft_17, dealer_no_blackjack,
            previous_depth_data
        )

    bust_counts = defaultdict(int)
    stand_counts = defaultdict(int)
    other_counts = defaultdict(int)

    for current_depth in range(1, depth):  # stops at depth-1
        for combo in product(range(2, 12), repeat=current_depth):
            if upcard + combo[0] == 21 and dealer_no_blackjack:
                continue
            hand = ValueOnlyHand([upcard])
            stopped_early = False

            for v in combo:
                if _dealer_stand_or_bust(hand, dealer_hit_soft_17):
                    stopped_early = True
                    break
                hand.add_card(v)
            if stopped_early:
                continue

            final_combo = get_final_combo(hand.cards[1:], upcard, dealer_no_blackjack)
            if hand.is_bust():
                bust_counts[final_combo] += 1
            elif _dealer_stand_or_bust(hand, dealer_hit_soft_17):
                stand_counts[final_combo] += 1

    for combo in product(range(2, 12), repeat=depth):
        if upcard + combo[0] == 21 and dealer_no_blackjack:
            continue
        hand = ValueOnlyHand([upcard])
        stopped_early = False

        for v in combo:
            if _dealer_stand_or_bust(hand, dealer_hit_soft_17):
                stopped_early = True
                break
            hand.add_card(v)
        if stopped_early:
            continue

        final_combo = get_final_combo(hand.cards[1:], upcard, dealer_no_blackjack)
        if hand.is_bust():
            bust_counts[final_combo] += 1
        elif _dealer_stand_or_bust(hand, dealer_hit_soft_17):
            stand_counts[final_combo] += 1
        else:
            other_counts[final_combo] += 1

    return _format_output(bust_counts, stand_counts, other_counts, upcard)


def _format_output(bust_counts, stand_counts, other_counts, upcard):
    bust_combos = sorted(bust_counts.keys())
    bust_combos_data = [(comb, bust_counts[comb]) for comb in bust_combos]

    stand_combos = sorted(stand_counts.keys())
    stand_combos_data = [(comb, stand_counts[comb]) for comb in stand_combos]
    stand_values = np.array([
        ValueOnlyHand([upcard, *comb]).get_best_value() for comb in stand_combos
    ])

    stand_combos = sorted(other_counts.keys())
    other_combos_data = [(comb, other_counts[comb]) for comb in other_counts]

    return dict(
        bust_combos_data=bust_combos_data, 
        stand_combos_data=stand_combos_data, 
        stand_values=stand_values, 
        other_combos_data=other_combos_data,
        bust_counts=bust_counts,
        stand_counts=stand_counts,
        other_counts=other_counts
    )

In [432]:

precomputed_combinations_with_counts_no_bj = dict()
precomputed_combinations_with_counts = dict()

for depth in range(1, 6):
    print(depth)
    precomputed_combinations_with_counts[depth] = dict()
    precomputed_combinations_with_counts_no_bj[depth] = dict()
    for upcard in range(2, 12):
        precomputed_combinations_with_counts_no_bj[depth][upcard] = get_realistic_combinations(
            depth, upcard, dealer_hit_soft_17=False, dealer_no_blackjack=True
        )
        precomputed_combinations_with_counts[depth][upcard] = get_realistic_combinations(
            depth, upcard, dealer_hit_soft_17=False, dealer_no_blackjack=False
        )

1
2
3
4
5


In [449]:
_combinations_with_counts_no_bj = dict()
_combinations_with_counts = dict()

for depth in range(1, 9):
    print(depth)
    _combinations_with_counts[depth] = dict()
    _combinations_with_counts_no_bj[depth] = dict()
    for upcard in range(2, 12):
        _combinations_with_counts[depth][upcard] = get_realistic_combinations(
            depth, upcard, dealer_hit_soft_17=False, dealer_no_blackjack=True,
            previous_depth_data=_combinations_with_counts.get(depth-1, {}).get(upcard, None)
        )
        _combinations_with_counts_no_bj[depth][upcard] = get_realistic_combinations(
            depth, upcard, dealer_hit_soft_17=False, dealer_no_blackjack=True,
            previous_depth_data=_combinations_with_counts_no_bj.get(depth-1, {}).get(upcard, None)
        )

1
2
3
4
5


KeyboardInterrupt: 

In [ ]:
precomputed_combinations_with_counts_no_bj = _combinations_with_counts_no_bj
precomputed_combinations_with_counts = _combinations_with_counts

In [450]:
import pickle


# with open("combinations/combinations_with_counts_no_bj.pkl", "wb") as f:
#     pickle.dump(
#         precomputed_combinations_with_counts_no_bj,
#         f,
#         protocol=pickle.HIGHEST_PROTOCOL,
#     )

# with open("combinations/combinations_with_counts.pkl", "wb") as f:
#     pickle.dump(
#         precomputed_combinations_with_counts_no_bj,
#         f,
#         protocol=pickle.HIGHEST_PROTOCOL,
#     )

In [451]:
with open("combinations/combinations_with_counts_no_bj.pkl", "rb") as f:
    precomputed_combinations_with_counts_no_bj = pickle.load(f)

with open("combinations/combinations_with_counts.pkl", "rb") as f:
    precomputed_combinations_with_counts = pickle.load(f)

In [431]:
for depth in range(1, 6):
    for upcard in range(2, 12):
        for k in _combinations_with_counts_no_bj[depth][upcard].keys():
            k_equal = np.all(
                _combinations_with_counts_no_bj[depth][upcard][k]
                == precomputed_combinations_with_counts_no_bj[depth][upcard][k]
            )
            print(
                k,
                k_equal   
            )
            assert k_equal

bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand_combos_data True
stand_values True
other_combos_data True
bust_counts True
stand_counts True
other_counts True
bust_combos_data True
stand

In [ ]:
import itertools


def get_cards_probability(cards, shoe, possible_first_card=None):
    prob = 1
    for i, card in enumerate(cards):
        if i == 0:
            possible_cards = possible_first_card
        else:
            possible_cards = None
        probs = shoe.get_rank_value_probabilities(possible_cards)
        if card not in probs or probs[card] == 0:
            return 0
        prob = prob * probs[card]
        shoe.burn_rank_value(card)
    for card in cards:
        shoe.add_rank_value(card)
    return prob


def run_dealer_cards_simulation_comb(
    bj_round: BJRound,
    shoe: ProbabilisticRankShoe,
    n_dealer_sim_runs: int = 1,
    n_full_sample: int = 4,
    verbose = False
):
    dealer_hand: ValueOnlyHand = bj_round.dealer_hand.copy()
    dealer_hit_soft_17 = bj_round.rules.dealer_hits_soft_17
    assert dealer_hand.size() == 1
    assert len(bj_round.player_hands) == 1
    assert bj_round.get_stage() == BJStage.DEALER_CARD
    assert not bj_round.player_hands[0].is_natural_blackjack()


    if shoe.is_dealer_card_locked():
        shoe = shoe.copy()
        shoe.unlock_dealer_card()
        
    player_hand_value = bj_round.player_hands[0].get_best_value()

    assert not bj_round.rules.dealer_hits_soft_17
    if bj_round.rules.dealer_checks_blackjack:
        assert not bj_round.dealer_has_bj_after_check
        data = precomputed_combinations_with_counts_no_bj[n_full_sample][dealer_hand.cards[0]]
    else:
        data = precomputed_combinations_with_counts[n_full_sample][dealer_hand.cards[0]]
        
    possible_second_card = bj_round.get_possible_next_card_ranks()

    bust_combos_data = data['bust_combos_data']
    stand_combos_data = data['stand_combos_data']
    other_combos_data = data['other_combos_data']
    stand_values = data['stand_values']

    total_p = 0 

    dealer_bust_p = 0
    for i, (combo, num_perms) in enumerate(bust_combos_data):
        prob = get_cards_probability(combo, shoe, possible_second_card)
        if prob != 0:
            prob = prob * num_perms
            total_p += prob
            dealer_bust_p += prob
        
    ev_stand = np.zeros(len(stand_combos_data))
    prob_stand = np.zeros(len(stand_combos_data))
    ev_stand[player_hand_value > stand_values] = 1
    ev_stand[player_hand_value == stand_values] = 0
    ev_stand[player_hand_value < stand_values] = -1
    for i, (combo, num_perms) in enumerate(stand_combos_data):
        prob = get_cards_probability(combo, shoe, possible_second_card)
        if prob != 0:
            prob = prob * num_perms
            total_p += prob
            prob_stand[i] = prob

    ev_other = np.zeros(len(stand_combos_data))
    prob_other= np.zeros(len(stand_combos_data))
    
    for i, (combo, num_perms) in enumerate(other_combos_data):
        prob = 1
        impossible = False
        burned_cards = []
        for i, card in enumerate(combo):
            if i == 0:
                probs = shoe.get_rank_value_probabilities(possible_second_card)
            else:
                probs = shoe.get_rank_value_probabilities()
            if card not in probs or probs[card] == 0:
                impossible = True
                break
            prob = prob * probs[card]
            shoe.burn_rank_value(card)
            burned_cards.append(card)
            dealer_hand.add_card(card)
  
        if impossible:
            for card in burned_cards:
                shoe.add_rank_value(card) 
                dealer_hand.pop_card()
            continue
        
        prob = prob * num_perms
        total_p += prob
        prob_other[i] = prob

        assert not _dealer_stand_or_bust(dealer_hand, dealer_hit_soft_17)
        sim_values = _run_dealer_cards_simulation(
            player_hand_value,
            dealer_hand,
            shoe,
            n_dealer_sim_runs,
            dealer_hit_soft_17
        )
        ev_other[i] = np.mean(sim_values)
        for card in burned_cards:
            shoe.add_rank_value(card)
            dealer_hand.pop_card()
    

    p_bust = dealer_bust_p
    ev_bust_value = 1 * dealer_bust_p
    p_stand = prob_stand.sum()
    p_other = total_p - p_bust - p_stand
    ev_stand_value = ev_stand.dot(prob_stand).item()
    ev_other_value = ev_other.dot(prob_other).item()
    if verbose:
        print("p_bust =", p_bust)
        print("p_stand =", p_stand, "ev_stand_value =", ev_stand_value)
        print("p_other =", p_other, "ev_other_value =", ev_other_value)
        print("p_total =", total_p)
    final_value = bj_round.bet_unit * (
        ev_other_value
        + ev_stand_value
        + ev_bust_value
    )

    return final_value

In [435]:
data = precomputed_combinations_with_counts[1][11]
bust_combos_data = data['bust_combos_data']
stand_combos_data = data['stand_combos_data']
stand_values = data['stand_values']
other_combos_data = data['other_combos_data']

In [436]:
other_combos_data[0]

((2,), 1)

In [437]:
# print(stand_values)

# print()
# print(bust_combos_data)
# print()
# print(stand_combos_data)
# print()
# print(other_combos_data)

In [438]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

In [452]:
bj_round = BJRound(rules)
shoe = ProbabilisticRankShoe.seeded(8, 42)
bj_round.start_round(10)

cards = [
    Card(Rank.ACE),
    Card(Rank.FIVE),
    Card(Rank.ACE) 
]

# cards = [
#     Card(Rank.NINE),
#     Card(Rank.SEVEN),
#     Card(Rank.SIX) 
# ]

cards = [c.rank_value() for c in cards]

bj_round.take_card(cards[0])
bj_round.take_card(cards[1])
bj_round.take_card(cards[2])

shoe.burn_rank_value(cards[0])
shoe.burn_rank_value(cards[1])
shoe.burn_rank_value(cards[2])

if PlayerAction.REFUSE_INSURANCE in bj_round.get_available_actions():
    bj_round.take_action(PlayerAction.REFUSE_INSURANCE)
if DealerAction.CONFIRM_NO_BLACKJACK in bj_round.get_available_actions():
    bj_round.take_action(DealerAction.CONFIRM_NO_BLACKJACK)
bj_round.take_action(PlayerAction.STAND)


In [453]:
value_rec = run_dealer_cards_simulation_recursive(
    bj_round,
    shoe,
    n_dealer_sim_runs=1,
    n_full_sample=5
)

In [ ]:
value_comb = run_dealer_cards_simulation_comb(
    bj_round,
    shoe,
    n_dealer_sim_runs=1,
    n_full_sample=7
)

p_bust = 0.16709298581403578
p_stand = 0.8329052841548049 ev_stand_value = -0.8329052841548049
p_other = 1.7300311597745477e-06
p_total = 1.0000000000000004


In [455]:
# total_p 1.000000000000002
# p_bust 0.1150213630730473
# p_stand 0.884947962554003 ev_stand -8.84947962554003
# p_other 3.067437295167785e-05

In [396]:
value_rec, value_comb

(np.float64(-6.658127952742149), -6.658029929874349)

In [383]:
values, dealer_bust_outcome, dealer_stand_outcome, dealer_other_outcome =\
    run_dealer_cards_simulation_by_outcome(
        bj_round,
        shoe,
        10000000,
        6
    )

In [349]:
print("p_bust", len(dealer_bust_outcome) / len(values))
print("p_stand", len(dealer_stand_outcome) / len(values))
print("p_other", len(dealer_other_outcome) / len(values))
print(np.mean(values))

p_bust 0.1670461
p_stand 0.8329539
p_other 0.0
-6.659078


In [16]:
dealer_other_outcome

[]

In [17]:
np.mean(dealer_stand_outcome)

np.float64(-10.0)

In [18]:
values

[10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 -10,
 10,
 10,
 -10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 -10,
 10,
 10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 10,
 -10,
 10,
 -10,
 10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 -10,
 10,
 -10,
 10,


In [19]:
expected = -0.154 * 10
print(expected) # 97 vs 6

-1.54


In [20]:
bj_round.get_stage()

<BJStage.DEALER_CARD: 7>

In [21]:
old_sim_values = []

n_sim_runs = 1000
 
for i in tqdm.tqdm(range(n_old_sim_runs)):
    old_simulation_value = \
        run_dealer_cards_simulation(bj_round, shoe, n_dealer_sim_runs=1000)
    old_sim_values.append(old_simulation_value)

NameError: name 'n_old_sim_runs' is not defined

In [ ]:
old_simulation_value = np.mean(old_sim_values)
old_simulation_value

In [294]:
values_rec_by_depth = dict()


n_sim_runs = 1000
for depth_i in range(1, 8):
    values_rec = []
    t0 = time.time()
    for i in tqdm.tqdm(range(n_sim_runs)):
        value_comb = run_dealer_cards_simulation_recursive(
            bj_round,
            shoe,
            n_dealer_sim_runs=1,
            n_full_sample=depth_i
        )
        values_rec.append(value_comb)
    values_rec_by_depth[depth_i] = values_rec
    t1 = time.time()
    print(f"Depth {depth_i} took {(t1 - t0) / n_sim_runs * 1000 } milliseconds")

100%|██████████| 1000/1000 [00:00<00:00, 12051.18it/s]


Depth 1 took 0.08838939666748047 milliseconds


100%|██████████| 1000/1000 [00:00<00:00, 3290.27it/s]


Depth 2 took 0.30576562881469727 milliseconds


100%|██████████| 1000/1000 [00:00<00:00, 1448.09it/s]


...

Depth 5 took 1.5634539127349854 milliseconds


100%|██████████| 1000/1000 [00:01<00:00, 629.43it/s]


Depth 6 took 1.591029405593872 milliseconds


100%|██████████| 1000/1000 [00:01<00:00, 597.31it/s]

Depth 7 took 1.6758527755737305 milliseconds


In [463]:
values_comb_by_depth = dict()


n_sim_runs = 1000
for depth_i in range(1, 8):
    values_comb = []
    t0 = time.time()
    for i in tqdm.tqdm(range(n_sim_runs)):
        value_comb = run_dealer_cards_simulation_comb(
            bj_round,
            shoe,
            n_dealer_sim_runs=1,
            n_full_sample=depth_i
        )
        values_comb.append(value_comb)
    t1 = time.time()
    print(f"Depth {depth_i} took {(t1 - t0) / n_sim_runs * 1000 } milliseconds")
    values_comb_by_depth[depth_i] = values_comb

100%|██████████| 1000/1000 [00:00<00:00, 5341.93it/s]


p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 ev_stand_value = -0.4491228070175438
p_other = 0.5508771929824562
p_total = 1.0
p_bust = 0
p_stand = 0.4491228070175438 

 17%|█▋        | 172/1000 [00:00<00:00, 1712.07it/s]

p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand_value = -0.6603815363651847
p_other = 0.33961846363481507
p_total = 0.9999999999999999
p_bust = 0
p_stand = 0.6603815363651848 ev_stand

 55%|█████▌    | 551/1000 [00:00<00:00, 1856.03it/s]

...

 46%|████▌     | 460/1000 [00:01<00:01, 322.98it/s]

p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911

 51%|█████     | 508/1000 [00:01<00:01, 318.32it/s]

p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911289
p_other = 0.013037179060197635
p_total = 1.0000000000000004
p_bust = 0.1594307650285138
p_stand = 0.827532055911289 ev_stand_value = -0.827532055911

KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt



sns.set_theme(style="whitegrid")  # optional styling

In [ ]:
f, ax = plt.subplots()


for i in range(3, 4):
    sns.histplot(values_comb_by_depth[i], bins=30, axes=ax, label=f"comb_{i}", stat="density")

for i in range(3, 4):
    sns.histplot(values_rec_by_depth[i], bins=30, axes=ax, label=f"rec_{i}", stat="density")


# sns.histplot(values_comb, bins=30, color="green", axes=ax, label="comb", stat="density")


# ax.axvline(
#     old_simulation_value,
#     color="red", linestyle="--", linewidth=2,
#     label=f"Mean = {old_simulation_value}"
# )



plt.legend()
plt.show()

In [ ]:
values_comb

In [ ]:
values_rec